# VAE's decoder architecture

In [1]:
import torch as th
import numpy as np
from einops import rearrange
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
from importlib import reload
import vae_decoder
from vae_decoder import unpatchify, Head
reload(vae_decoder)

# Dev
device = 'cuda'
J = 25
out_J_chn = 2
# 0 resblock = we have at least 1 resblock in the decoder (during upsampling) see vae_decoder.py
vae_decoder = vae_decoder.JointVAE38(J=J, out_J_chn=out_J_chn, z_dim=48, num_res_blocks=0).to(device)
# print(vae_decoder)
# for name, param in vae_decoder.named_parameters():
#     print(name, param.shape, param.requires_grad)

In [2]:
# data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_320p/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
# data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_480p/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
# data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_320p_45frames/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
data = th.load('/host/data2/mint/Motion_Dataset/SkelAg/frontview_bodyjoints_320p_9frames/dit_feats/0/0.pth', weights_only=False, map_location='cpu')
dit_features = data[0]['dit_features']
grid_size = data[0]['grid_size']
patch_size = data[0]['patch_size']
dim = data[0]['dim']
out_dim = data[0]['out_dim']
tiled = data[0]['tiled']
tile_size = data[0]['tile_size']
tile_stride = data[0]['tile_stride']
z_dim = data[0]['z_dim']  # vae z_dim
motion_gt_2d = data[0]['joints_2d']
motion_gt_3d = data[0]['joints_3d']
bones = data[0]['bones']
joint_names = data[0]['joint_names']
edges = [[joint_names.index(b[0]), joint_names.index(b[1])] for b in bones]
print("dit_features:", dit_features.shape)
print("num frames:", data[0]['num_frames'])
print("joints_3d:", data[0]['joints_3d'].shape)
print("joints_2d:", data[0]['joints_2d'].shape)
print("patch_size:", patch_size)
print("dim:", dim)
print("out_dim:", out_dim)
print("z_dim:", z_dim)
print("tiled:", tiled)
print("tile_size:", tile_size)
print("tile_stride:", tile_stride)
print(data[0].keys())
print(dit_features.shape, grid_size)
print(dit_features[0:1].shape, grid_size[0], grid_size[1], grid_size[2])

from diffsynth.diffusion.vis import MultiSkeleton2D3DAnimator
anim = MultiSkeleton2D3DAnimator(fps=30, title="Motions", y_axis_down=True)
anim.add_sequence(motion_gt_3d, K2=motion_gt_2d[..., :2], edges=edges, color="blue", name="Ground Truth")



dit_features: torch.Size([30, 1, 600, 3072])
num frames: 9
joints_3d: (9, 25, 3)
joints_2d: (9, 25, 3)
patch_size: [1, 2, 2]
dim: 3072
out_dim: 48
z_dim: 48
tiled: True
tile_size: (30, 52)
tile_stride: (15, 26)
dict_keys(['input_video', 'height', 'width', 'num_frames', 'cfg_scale', 'tiled', 'tile_size', 'tile_stride', 'rand_device', 'use_gradient_checkpointing', 'use_gradient_checkpointing_offload', 'cfg_merge', 'vace_scale', 'max_timestep_boundary', 'min_timestep_boundary', 'preferred_timestep_id', 'preferred_dit_block_id', 'joints_3d', 'joints_2d', 'cams_intr', 'cams_extr', 'joint_names', 'bones', 'input_image', 'noise', 'latents', 'input_latents', 'fuse_vae_embedding_in_latents', 'first_frame_latents', 'vace_context', 'animate_pose_video', 'animate_face_video', 'animate_inpaint_video', 'animate_mask_video', 'prompt', 'context', 'dit_features', 'grid_size', 'dim', 'out_dim', 'patch_size', 'z_dim'])
torch.Size([30, 1, 600, 3072]) (3, 10, 20)
torch.Size([1, 1, 600, 3072]) 3 10 20


In [ ]:
def unpatchify(x, grid_size, patch_size):
    patch_size = [1, 2, 2]
    return rearrange(
        x, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
        f=grid_size[0], h=grid_size[1], w=grid_size[2], 
        x=patch_size[0], y=patch_size[1], z=patch_size[2]
    )
    
# 3072 48 [1, 2, 2] 1e-06
dim = 3072
out_dim = 48
patch_size = [1, 2, 2]
eps = 1e-6

inp = dit_features[0].type(th.float32).to(device)
head = Head(dim=dim, out_dim=out_dim, patch_size=patch_size, eps=eps).to(device)
print("Input: ", inp.shape)
out_head = head(inp)
print("Out head: ", out_head.shape)
out_unpatched = unpatchify(out_head, grid_size)
print("Out unpatched: ", out_unpatched.shape)
# assert False
out_tiled = vae_decoder.decode(out_unpatched[:, :, :4, :, :], device=device, tiled=tiled, tile_size=tile_size, tile_stride=tile_stride).cpu()
print(out_tiled.shape)
# out_notiled = vae_decoder.decode(out_unpatched[:, :, :2, :, :], device=device).cpu()
# print(out_notiled.shape)
# out_diff = th.abs(out_tiled - out_notiled)
# print("Diff tiled vs notiled: ", out_diff.max(), out_diff.mean())

# Imitate training loops

In [ ]:
import tqdm
T = 2
H = 320
W = 640
eps = 1e-6
gt = th.zeros(1, 3, 4 * T + 1, H, W)
inp = dit_features[0].type(th.float32).to(device)

def unpatchify(x, grid_size):
    patch_size = [1, 2, 2]
    return rearrange(
        x, 'b (f h w) (x y z c) -> b c (f x) (h y) (w z)',
        f=grid_size[0], h=grid_size[1], w=grid_size[2], 
        x=patch_size[0], y=patch_size[1], z=patch_size[2]
    )

head = Head(dim=dim, out_dim=out_dim, patch_size=patch_size, eps=eps).to(device)
optimizer = th.optim.Adam(list(head.parameters()) + list(vae_decoder.parameters()), lr=1e-4)
    
for name, param in head.named_parameters():
    print(name, param.shape, param.requires_grad)
    
def check_params_stats(model):
    total_params = 0
    trainable_params = 0
    for param in model.parameters():
        num_params = param.numel()
        total_params += num_params
        if param.requires_grad:
            trainable_params += num_params
    return total_params, trainable_params

for name, param in vae_decoder.named_parameters():
    param.requires_grad = True
    # print(name, param.shape, param.requires_grad)
    
before_train_total, before_trainable = check_params_stats(vae_decoder)
print(f"Before training - Total parameters: {before_train_total}, Trainable parameters: {before_trainable}")
print(f"Stats (mean/std) of parameters before training:", th.tensor([p.data.mean().item() for p in vae_decoder.parameters() if p.requires_grad]).mean().item(), th.tensor([p.data.std().item() for p in vae_decoder.parameters() if p.requires_grad]).mean().item())
    
    
loss_list = []
t = tqdm.trange(100)
for i in t:
    vae_decoder.model.clear_cache()
    optimizer.zero_grad()
    print(inp.shape)
    assert False
    out_head = head(inp)
    # print(out_head.shape)
    out_unpatched = unpatchify(out_head, grid_size)
    # print(out_unpatched.shape)
    # out_tiled = vae_decoder.decode(out_unpatched[:, :, :T+1, :, :], device=device, tiled=tiled, tile_size=tile_size, tile_stride=tile_stride)
    out_tiled = vae_decoder.decode(out_unpatched[:, :, :, :, :], device=device)
    # print(out_tiled.shape, gt.shape)
    loss = th.nn.functional.mse_loss(out_tiled.to(device), gt.to(device))
    loss.backward()
    optimizer.step()
    t.set_description(f"Loss: {loss.item():.6f}")
    loss_list.append(loss.item())
    
    
after_train_total, after_trainable = check_params_stats(vae_decoder)
print(f"After training - Total parameters: {after_train_total}, Trainable parameters: {after_trainable}")
print(f"Stats (mean/std) of parameters after training:", th.tensor([p.data.mean().item() for p in vae_decoder.parameters() if p.requires_grad]).mean().item(), th.tensor([p.data.std().item() for p in vae_decoder.parameters() if p.requires_grad]).mean().item())